# Notebook for selecting additional proteins for experimental validation

In [ ]:
import pandas as pd 
import os
rewrite = False

repo_path = os.path.dirname(os.getcwd())
repo_path = repo_path.split("WP2_Sweet_Protein_Discovery")[0] + "/WP2_Sweet_Protein_Discovery"

# Read proteins from previous quote
previous_candidates_df = pd.read_csv(os.path.join(repo_path, "8_protein_selection", "3_genscript_quote_id_mapping.csv"))

peptide_name_abbreviations = {
    "MNEI_1IV7": "MN",
    "Honey_truffle_sweetener": "HTS",
    "Honey_truffle_sweetener_native": "HTS_native"
}
final_cols = ['original_id', 'rank', 'protein_name', 'sequence', 'cysteine_count', 'disulfide_bridges']

# Load 0_de_novo_candidates.tsv, 0_foldseek_candidates.tsv and all_candidates.tsv
all_candidates = pd.read_csv(os.path.join(repo_path, "8_protein_selection", "5_2nd_selection_round", "1_merged_candidates.tsv"), sep = "\t")

# The following proteins have identical sequences to other candidates
all_candidates = all_candidates[all_candidates["target_id"] != "OneKP_scaffold-IXLH-2105913-Polypodium_hesperium-B_type"]
all_candidates = all_candidates[all_candidates["target_id"] != "OneKP_scaffold-RCBT-2003187-Sphagnum_palustre"]
all_candidates = all_candidates[all_candidates["target_id"] != "OneKP_scaffold-BMIF-2000308-Odontosororia_sp."]
all_candidates = all_candidates[~all_candidates["sequence"].isin(previous_candidates_df["Sequence"])]

# Using results from 0.5 sequence identity clustering using MMseqs2 
seq_id = "0.5"

# MNEI
# Read clustered sequences and get the highest similarity hit for each cluster.
# Removing cluster containing the de novo sequences (fixed_ and free_) to avoid biasing the selection towards de novo candidates.
mnei_clusters_df = pd.read_csv(f"{repo_path}/8_protein_selection/5_2nd_selection_round/2_mmseqs_clustered_hits/MNEI_1IV7/MNEI_1IV7_{seq_id}_cluster.tsv", sep="\t", header=None, names=["cluster_id", "target_id"])
mnei_clusters_df = mnei_clusters_df.drop_duplicates()
mnei_clusters_df = mnei_clusters_df.merge(all_candidates, on = "target_id", how = "inner")
mnei_clusters_df = mnei_clusters_df.sort_values(by=['cluster_id', 'avg_sweet_res_similarity'], ascending=[True, False])
mnei_clusters_df['representative_hit'] = mnei_clusters_df.groupby('cluster_id')['target_id'].transform('first')
mnei_clusters_df = mnei_clusters_df[~mnei_clusters_df["representative_hit"].str.startswith(("fixed", "free"))]
mnei_clusters_df = mnei_clusters_df.drop(columns=['cluster_id'])
print("MNEI has " + str(mnei_clusters_df["representative_hit"].nunique()) + " unique representative hits.")

# Selecting the top 10 clusters based on avg_sweet_res_similarity and keeping only the representative hit for each cluster
# based on the similarity of sweet residues
mnei_selected_df = mnei_clusters_df.sort_values(by='avg_sweet_res_similarity', ascending=False).groupby('representative_hit').head(5).head(30)
mnei_selected_df["rank"] = mnei_selected_df["avg_sweet_res_similarity"].rank(ascending=False)
mnei_selected_df["rank"] = mnei_selected_df["rank"] + 10
mnei_selected_df

MNEI has 28 unique representative hits.


,target_id,query,of3_tm_score,of3_RMSD,of3_tm_align_start,of3_tm_align_end,of3_avg_pLDDT,of3_tm_align_seq_id,of3_tm_align_length,source,...,sequence,best_model,similarity_scores,avg_surfdiff_similarity,avg_sweet_res_similarity,cysteine_count,disulfide_bridges,all_cysteines_paired,representative_hit,rank
117,A0AAD5CWR7,MNEI_1IV7,0.80041,1.37,0.0,99.0,84.451800,0.282,85.0,UniProt,...,GERTKVDDVKTNKAIQEIGRYCVDEYNRLPRSIVANGDGALRFSRV...,MNEI_1IV7___A0AAD5CWR7_seed_42_sample_4_model.pdb,"[0.440263683187032, 0.4523130230186538, 0.4693...",0.537664,0.601203,1.0,[],False,A0AAD5CWR7,11.0
139,A0A835F440,MNEI_1IV7,0.75462,1.59,0.0,89.0,87.874444,0.305,82.0,UniProt,...,GGWSPIKDVSDPYIQELGAWAVSEYLKQGHVGGLQYGQVLSGEQQV...,MNEI_1IV7___A0A835F440_seed_42_sample_3_model.pdb,"[0.5223315358881029, 0.596558529093747, 0.4971...",0.510483,0.590584,0.0,[],True,A0A835F440,12.0
130,A0A2S3IPD9,MNEI_1IV7,0.78335,1.50,0.0,92.0,88.960215,0.318,85.0,UniProt,...,GGWSAIPDVSDPRIQELGEWALGQAKRARLAGEGLRFRRVVRGEQQ...,MNEI_1IV7___A0A2S3IPD9_seed_42_sample_5_model.pdb,"[0.7891054374679052, 0.5985806419192896, 0.455...",0.612035,0.570532,0.0,[],True,A0A835F440,13.0
103,A0AAD5GGF7,MNEI_1IV7,0.78888,2.08,0.0,98.0,83.334700,0.267,86.0,UniProt,...,GGRTKIDDVKTNKEIQAIGRYSVDEYNRLHGSGNNDGELTFSQVVE...,MNEI_1IV7___A0AAD5GGF7_seed_42_sample_3_model.pdb,"[0.3277937743944695, 0.3994635882900279, 0.397...",0.536811,0.567012,0.0,[],True,A0AAD5CWR7,14.0
0,I1P8S2,MNEI_1IV7,0.73725,1.92,0.0,88.0,84.726854,0.280,82.0,UniProt,...,ESWKTIDANDRHVQDVALWAVAETDWASATGGLTLNTVDGAEKRFE...,MNEI_1IV7___I1P8S2_seed_42_sample_5_model.pdb,"[0.0414187490544487, 0.1922815867969007, 0.375...",0.476470,0.563944,0.0,[],True,I1P8S2,15.0
1,A0A0D3FFA5,MNEI_1IV7,0.73744,1.92,0.0,88.0,84.870000,0.280,82.0,UniProt,...,ESWKTIDANDRHVQDVALWAVAETDWASATGGLTLNTVDGAEKRFE...,MNEI_1IV7___A0A0D3FFA5_seed_42_sample_5_model.pdb,"[0.0414187490544487, 0.1922815867969007, 0.374...",0.478028,0.563372,0.0,[],True,I1P8S2,16.0
2,OneKP_scaffold-OLXF-2023012-Hibiscus_cannabinus,MNEI_1IV7,0.77795,1.68,0.0,90.0,86.756344,0.274,84.0,OneKP,...,GGISQVEGSANSLEFENLARFAVDEHNKKGNAMLQFKKVTNVKQQV...,MNEI_1IV7___OneKP_scaffold-OLXF-2023012-Hibisc...,"[0.1212671406483061, 0.1622185158916793, 0.208...",0.482401,0.563228,0.0,[],True,OneKP_scaffold-OLXF-2023012-Hibiscus_cannabinus,17.0
104,A0A103Y0M8,MNEI_1IV7,0.80692,1.40,0.0,99.0,87.271800,0.294,85.0,UniProt,...,GGRSKITDVNTNKEIQDLGRYSVEEYNRLHRSGTVKNGGDLTFYRV...,MNEI_1IV7___A0A103Y0M8_seed_42_sample_4_model.pdb,"[0.3817375386819005, 0.4108475289858394, 0.422...",0.537636,0.562978,0.0,[],True,A0AAD5CWR7,18.0
121,A0AAV0RKK3,MNEI_1IV7,0.76615,1.68,0.0,92.0,87.345806,0.250,84.0,UniProt,...,WEWEPIKNLKDEHVQEIAEYAVQAYNMISNSFPIQLVKVEKGKIRY...,MNEI_1IV7___A0AAV0RKK3_seed_42_sample_2_model.pdb,"[0.2596171782206972, 0.4642719012720151, 0.657...",0.505360,0.562946,0.0,[],True,A0AAV0RKK3,19.0
116,A0A251TD67,MNEI_1IV7,0.78780,1.52,0.0,98.0,80.187745,0.226,84.0,UniProt,...,LGARTKVDHVKTNKMIQEIGRYSVDEYNRLPRSTTANDEGGLRFSR...,MNEI_1IV7___A0A251TD67_seed_42_sample_2_model.pdb,"[0.279730706199483, 0.3280957095812675, 0.4613...",0.528432,0.562092,0.0,[],True,A0AAD5CWR7,20.0


In [3]:
# HTS
# Read clustered sequences and get the highest similarity hit for each cluster.
hts_clusters_df = pd.read_csv(f"{repo_path}/8_protein_selection/5_2nd_selection_round/2_mmseqs_clustered_hits/Honey_truffle_sweetener/Honey_truffle_sweetener_{seq_id}_cluster.tsv", sep="\t", header=None, names=["cluster_id", "target_id"])
hts_clusters_df = hts_clusters_df.drop_duplicates()
hts_clusters_df = hts_clusters_df.merge(all_candidates, on = "target_id", how = "inner")
hts_clusters_df = hts_clusters_df.sort_values(by=['cluster_id', 'avg_sweet_res_similarity'], ascending=[True, False])
hts_clusters_df['representative_hit'] = hts_clusters_df.groupby('cluster_id')['target_id'].transform('first')
hts_clusters_df = hts_clusters_df.drop(columns=['cluster_id'])
print("Honey_truffle_sweetener has " + str(hts_clusters_df["representative_hit"].nunique()) + " unique representative hits.")

hts_selected_df = hts_clusters_df.sort_values(by='avg_sweet_res_similarity', ascending=False).groupby('representative_hit').head(1).head(11)
hts_selected_df["rank"] = hts_selected_df["avg_sweet_res_similarity"].rank(ascending=False)
hts_selected_df["rank"] = hts_selected_df["rank"] + 10

# Concatenate the selected candidates for MNEI and Thaumatin and save to a new dataframe
final_representatives = pd.concat([mnei_selected_df, hts_selected_df])

final_representatives["rank"] = final_representatives["rank"].astype(int)
final_representatives["protein_name"] = final_representatives.apply(lambda row: peptide_name_abbreviations[row["query"]] + "_" + str(int(row["rank"])), axis = 1)
final_representatives = final_representatives.rename(columns={"target_id": "original_id"})
final_representatives = final_representatives[final_cols].copy()
if rewrite:
    final_representatives.to_csv(os.path.join(repo_path, "8_protein_selection", "4_additional_candidates.tsv"), sep = "\t", index = False)


Honey_truffle_sweetener has 32 unique representative hits.


In [5]:
final_representatives

,original_id,rank,protein_name,sequence,cysteine_count,disulfide_bridges
117,A0AAD5CWR7,11,MN_11,GERTKVDDVKTNKAIQEIGRYCVDEYNRLPRSIVANGDGALRFSRV...,1.0,[]
139,A0A835F440,12,MN_12,GGWSPIKDVSDPYIQELGAWAVSEYLKQGHVGGLQYGQVLSGEQQV...,0.0,[]
130,A0A2S3IPD9,13,MN_13,GGWSAIPDVSDPRIQELGEWALGQAKRARLAGEGLRFRRVVRGEQQ...,0.0,[]
103,A0AAD5GGF7,14,MN_14,GGRTKIDDVKTNKEIQAIGRYSVDEYNRLHGSGNNDGELTFSQVVE...,0.0,[]
0,I1P8S2,15,MN_15,ESWKTIDANDRHVQDVALWAVAETDWASATGGLTLNTVDGAEKRFE...,0.0,[]
1,A0A0D3FFA5,16,MN_16,ESWKTIDANDRHVQDVALWAVAETDWASATGGLTLNTVDGAEKRFE...,0.0,[]
2,OneKP_scaffold-OLXF-2023012-Hibiscus_cannabinus,17,MN_17,GGISQVEGSANSLEFENLARFAVDEHNKKGNAMLQFKKVTNVKQQV...,0.0,[]
104,A0A103Y0M8,18,MN_18,GGRSKITDVNTNKEIQDLGRYSVEEYNRLHRSGTVKNGGDLTFYRV...,0.0,[]
121,A0AAV0RKK3,19,MN_19,WEWEPIKNLKDEHVQEIAEYAVQAYNMISNSFPIQLVKVEKGKIRY...,0.0,[]
116,A0A251TD67,20,MN_20,LGARTKVDHVKTNKMIQEIGRYSVDEYNRLPRSTTANDEGGLRFSR...,0.0,[]


In [6]:
# Concatenate the selected representatives from both approaches
quote_proteins = final_representatives.copy()

# Read original sequences
foldseek_uniprot_df = pd.read_csv(f"{repo_path}/3_foldseek_search/foldseek_UniProt_search.tsv", sep = "\t")
foldseek_uniprot_df["target"] = foldseek_uniprot_df["target"].apply(lambda x: x.split("|")[1] if "|" in x else x)
foldseek_onekp_df = pd.read_csv(f"{repo_path}/3_foldseek_search/foldseek_OneKP_search.tsv", sep = "\t")

# Keep only target and tseq
foldseek_uniprot_df = foldseek_uniprot_df[["target", "tseq"]].copy()
foldseek_onekp_df = foldseek_onekp_df[["target", "tseq"]].copy()

# Merge the two foldseek results to get the original sequences for all candidates
foldseek_df = pd.concat([foldseek_uniprot_df, foldseek_onekp_df]).drop_duplicates(subset=["target"]).copy()
quote_proteins = quote_proteins.merge(foldseek_df, left_on="original_id", right_on="target", how="left").drop(columns=["target"]).rename(columns={"tseq": "full_sequence"})

# Peptide name = peptide_name_abbreviation + rank 
rename_cols = {
    "protein_name": "Protein_ID",
    "cysteine_count": "Cysteine_Count",
    "sequence": "Sequence",
    "disulfide_bridges": "Disulfide_Bridges",
    "original_id": "Original_ID",
    "full_sequence": "Full Sequence"
}

quote_proteins = quote_proteins.rename(columns = rename_cols)
quote_proteins = quote_proteins[rename_cols.values()]

# Save fasta file of the selected candidates for synthesis
if rewrite:
    fasta_path = os.path.join(repo_path, "8_protein_selection", "4_quote_full_sequences.fasta")
    if os.path.exists(fasta_path):
        os.remove(fasta_path)
    for index, row in quote_proteins.iterrows():
        fasta_header = f">{row['Protein_ID']}"
        if pd.isna(row["Full Sequence"]):
            fasta_sequence = row["Sequence"]
        else:
            fasta_sequence = row["Full Sequence"]
        fasta_entry = f"{fasta_header}\n{fasta_sequence}\n"
        with open(fasta_path, "a") as fasta_file:
            fasta_file.write(fasta_entry)

# Add DeepLoc 2 predictions to the resulting dataframe
DeepLoc 2 was run using the following command:
```
deeploc2 -f 4_quote_full_sequences.fasta -m Fast -o ./4_deeploc2_results
```

In [ ]:
deeploc_res = f"{repo_path}/8_protein_selection/5_2nd_selection_round/4_deeploc2_results/results_20260928-210043.csv"
deeploc_df = pd.read_csv(deeploc_res)
deeploc_df = deeploc_df[deeploc_df["Protein_ID"].isin(quote_proteins["Protein_ID"])]
deeploc_df = deeploc_df[["Protein_ID", "Localizations", "Membrane types", "Signals"]]
deeploc_df = deeploc_df.rename(columns={
    "Membrane types": "Membrane_Types"
})

# Merge to quote_proteins to get the localization information for each candidate
quote_proteins_final = quote_proteins.merge(deeploc_df, on="Protein_ID", how="inner")
if rewrite:
    quote_proteins_final[["Protein_ID", "Original_ID", "Sequence"]].to_csv(os.path.join(repo_path, "8_protein_selection", "5_2nd_selection_round", "3_genscript_quote_id_mapping.csv"), index = False)
quote_proteins_final_df = quote_proteins_final.drop(columns=["Original_ID"])

quote_proteins_final_df["Expressed_Sequence"] = ""
for index, row in quote_proteins_final_df.iterrows():
    # If the length of Original Sequence differs from full length sequence by less then 5 amino acids,
    # use the full length sequence for synthesis, otherwise use the original sequence (which is the sequence of the aligned region to the query).
    if pd.isna(row["Full Sequence"]):
        quote_proteins_final_df.at[index, "Expressed_Sequence"] = row["Sequence"]
    else:
        if abs(len(row["Full Sequence"]) - len(row["Sequence"])) <= 5:
            quote_proteins_final_df.at[index, "Expressed_Sequence"] = row["Full Sequence"]
        else:
            quote_proteins_final_df.at[index, "Expressed_Sequence"] = row["Sequence"]

if rewrite:
    quote_proteins_final_df[["Protein_ID", "Expressed_Sequence"]].to_csv(os.path.join(repo_path, "8_protein_selection", "5_2nd_selection_round", "3_genscript_quote.csv"), index = False)
quote_proteins_final_df

,Protein_ID,Cysteine_Count,Sequence,Disulfide_Bridges,Full Sequence,Localizations,Membrane_Types,Signals,Expressed_Sequence
0,MN_11,1.0,GERTKVDDVKTNKAIQEIGRYCVDEYNRLPRSIVANGDGALRFSRV...,[],MTNLFTIITFAIVILISGYVANGMLGERTKVDDVKTNKAIQEIGRY...,Extracellular,Soluble,Signal peptide,GERTKVDDVKTNKAIQEIGRYCVDEYNRLPRSIVANGDGALRFSRV...
1,MN_12,0.0,GGWSPIKDVSDPYIQELGAWAVSEYLKQGHVGGLQYGQVLSGEQQV...,[],MDSLTKLVSQRAVVIFDMSSGDKCHAATDMFRNLGVSPTVVELDKD...,Extracellular,Soluble,Signal peptide,GGWSPIKDVSDPYIQELGAWAVSEYLKQGHVGGLQYGQVLSGEQQV...
2,MN_13,0.0,GGWSAIPDVSDPRIQELGEWALGQAKRARLAGEGLRFRRVVRGEQQ...,[],MARPLLPLLLLLAALAAAAAFAVPARAALGGGGGGTGPRVGGWSAI...,Extracellular,Soluble,Signal peptide,GGWSAIPDVSDPRIQELGEWALGQAKRARLAGEGLRFRRVVRGEQQ...
3,MN_14,0.0,GGRTKIDDVKTNKEIQAIGRYSVDEYNRLHGSGNNDGELTFSQVVE...,[],MANHSITTITFTLVLLISVSSSVATGRLGGRTKIDDVKTNKEIQAI...,Extracellular,Soluble,Signal peptide,GGRTKIDDVKTNKEIQAIGRYSVDEYNRLHGSGNNDGELTFSQVVE...
4,MN_15,0.0,ESWKTIDANDRHVQDVALWAVAETDWASATGGLTLNTVDGAEKRFE...,[],MRTSSLVLFAAVAVFGAACTAAAGDESWKTIDANDRHVQDVALWAV...,Extracellular,Soluble,Signal peptide,ESWKTIDANDRHVQDVALWAVAETDWASATGGLTLNTVDGAEKRFE...
5,MN_16,0.0,ESWKTIDANDRHVQDVALWAVAETDWASATGGLTLNTVDGAEKRFE...,[],MRTSSLVLFAAVAVFGAACTAAAGDESWKTIDANDRHVQDVALWAV...,Extracellular,Soluble,Signal peptide,ESWKTIDANDRHVQDVALWAVAETDWASATGGLTLNTVDGAEKRFE...
6,MN_17,0.0,GGISQVEGSANSLEFENLARFAVDEHNKKGNAMLQFKKVTNVKQQV...,[],EMAKVGGISQVEGSANSLEFENLARFAVDEHNKKGNAMLQFKKVTN...,Cytoplasm|Nucleus,Soluble,Nuclear localization signal,GGISQVEGSANSLEFENLARFAVDEHNKKGNAMLQFKKVTNVKQQV...
7,MN_18,0.0,GGRSKITDVNTNKEIQDLGRYSVEEYNRLHRSGTVKNGGDLTFYRV...,[],MIMDKDSWKRGTPLVLPHPHFHLRVRHFLPLHKPSNSEMNLHITMA...,Extracellular,Soluble,Signal peptide,GGRSKITDVNTNKEIQDLGRYSVEEYNRLHRSGTVKNGGDLTFYRV...
8,MN_19,0.0,WEWEPIKNLKDEHVQEIAEYAVQAYNMISNSFPIQLVKVEKGKIRY...,[],MTTVVVILSTVNATIGGWEWEPIKNLKDEHVQEIAEYAVQAYNMIS...,Extracellular,Soluble,Signal peptide,WEWEPIKNLKDEHVQEIAEYAVQAYNMISNSFPIQLVKVEKGKIRY...
9,MN_20,0.0,LGARTKVDHVKTNKMIQEIGRYSVDEYNRLPRSTTANDEGGLRFSR...,[],MANLSTLVTFAIVILISCFVANGMLGARTKVDHVKTNKMIQEIGRY...,Extracellular,Soluble,Signal peptide,LGARTKVDHVKTNKMIQEIGRYSVDEYNRLPRSTTANDEGGLRFSR...


In [ ]:
selected_df = pd.read_excel(f"{repo_path}/8_protein_selection/5_2nd_selection_round/4_protein_candidates.xlsx")

for index, row in selected_df.iterrows():
    for i2, r2 in previous_candidates_df.iterrows():
        if row["Sequence"] in r2["Sequence"]:
            print(f"Selected candidate {row['Protein_ID']} has identical sequence to previous candidate {r2['Protein_ID']}.")
        elif r2["Sequence"] in row["Sequence"]:
            print(f"Selected candidate {row['Protein_ID']} has identical sequence to previous candidate {r2['Protein_ID']}.")

# Make sure that Sequence is contained in Full sequence in all selected_df
for index, row in selected_df.iterrows():
    if not row["Full sequence"] or pd.isna(row["Full sequence"]):
        print(f"Selected candidate {row['Protein ID']} has no Full Sequence. Skipping check.")
        continue
    if row["Sequence"] not in row["Full sequence"]:
        print(f"Selected candidate {row['Protein_ID']} has Sequence not contained in Full Sequence.")

# Save full sequence into a fasta 
if rewrite:
    fasta_path = os.path.join(repo_path, "8_protein_selection", "5_2nd_selection_round", "3_quote_full_sequences.fasta")
    for index, row in selected_df.iterrows():
        fasta_header = f">{row['Protein ID']}"
        fasta_sequence = row["Full sequence"]
        fasta_entry = f"{fasta_header}\n{fasta_sequence}\n"
        with open(fasta_path, "a") as fasta_file:
            fasta_file.write(fasta_entry)


Selected candidate CAO6430196.1 has no Full Sequence. Skipping check.
Selected candidate KAM0940905.1 has no Full Sequence. Skipping check.
